# Machine Learning "by Hand" with PyTorch Tensors and Autograd

In this notebook we build models **from the ground up**. We will write the model, the loss function, and the parameter update rule ourselves. The only thing we let the library do is compute derivatives.

We'll build up in four steps:

1. **Linear regression** trained with (full-batch) **gradient descent**
2. **Logistic regression**, moving from batch gradient descent to **stochastic** and **mini-batch gradient descent**
3. A **neural network for regression**: stacking linear layers with nonlinear activations
4. A **neural network for classification**: same network, different output activation and loss

Every model follows the same recipe:

| Step | What happens |
|---|---|
| Forward pass | use the current parameters to compute predictions $\hat{y}$ |
| Loss | measure how wrong the predictions are, $J(\hat{y}, y)$ |
| Backward pass | compute $\partial J / \partial \theta$ for every parameter $\theta$ |
| Update | step downhill: $\theta \leftarrow \theta - \alpha\, \partial J / \partial \theta$ |

The next notebook replaces each of these hand-written pieces with PyTorch's built-in tools (`nn.Module`, `torch.optim`, `DataLoader`, ...). Seeing them by hand first makes it much clearer what those tools are actually doing.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

torch.manual_seed(42)   # reproducible random numbers
print("PyTorch version:", torch.__version__)

# Part 0: Just enough PyTorch

A PyTorch **tensor** is a multidimensional array, very much like a NumPy array. What makes it special for machine learning is **autograd**: if you create a tensor with `requires_grad=True`, PyTorch records every operation applied to it, and can then compute derivatives of the result with respect to it.

(The next notebook covers tensors in more depth. Here we need only the essentials.)

In [ ]:
a = torch.tensor([1.0, 2.0, 3.0])
M = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])

print(a.shape, M.shape)
print(M @ a)          # matrix-vector product
print(a.mean(), a.dtype)

## Automatic differentiation

Let's take the derivative of $f(x) = x^2$ at $x = 3$. We know $f'(x) = 2x$, so we expect 6.

* `requires_grad=True` tells PyTorch to track operations on `x`
* `f.backward()` computes $\partial f / \partial x$
* the result is stored in `x.grad`

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
y = torch.tensor(1.0, requires_grad=True)
f = x ** 2 + y**2
f.backward()
x.grad

In [ ]:
y.grad

### Derivative of a whole curve

Now take $\tanh(x)$ evaluated at 1000 points. `backward()` needs a single number (a scalar) to differentiate, so we differentiate the **sum** of the outputs. Because each output $\tanh(x_i)$ depends only on its own input $x_i$,

$$\frac{\partial}{\partial x_i} \sum_j \tanh(x_j) = \tanh'(x_i),$$

so `x.grad` ends up holding the derivative at every point.

In [ ]:
x = torch.linspace(-4, 4, 1000, requires_grad=True)
y = torch.tanh(x)
y.sum().backward()

fig, ax = plt.subplots()
ax.plot(x.detach(), y.detach(), label=r"$\tanh(x)$")
ax.plot(x.detach(), x.grad, label=r"$\frac{d}{dx}\tanh(x)$ via autograd")
ax.legend()
plt.show()

`.detach()` gives a copy of a tensor with gradient tracking turned off. Matplotlib and NumPy need that before they can use a tensor that has `requires_grad=True`.

## Gradient descent on a single number

Before fitting data, let's minimize $f(x) = (x - 3)^2$, whose minimum is at $x = 3$. This loop is the skeleton of **every** training loop in this notebook:

1. compute the function (forward pass)
2. `backward()` to get the gradient
3. update the parameter inside `torch.no_grad()`, so that the update itself is not tracked as part of the computation
4. reset the gradient to zero, because **PyTorch adds new gradients to `.grad`** rather than replacing them

In [ ]:
x = torch.tensor(0.0, requires_grad=True)
learning_rate = 0.1

for step in range(30):
    f = (x - 3) ** 2          # 1. forward
    f.backward()              # 2. backward: x.grad = df/dx
    with torch.no_grad():
        x -= learning_rate * x.grad   # 3. update
    x.grad.zero_()            # 4. reset the gradient
    if step % 5 == 0:
        print(f"step {step:2d}   x = {x.item():.4f}   f(x) = {f.item():.4f}")

print("final x:", x.item())

> **Try it:** comment out `x.grad.zero_()` and rerun. The gradients pile up from step to step, and `x` overshoots and oscillates instead of settling at 3.

# Part 1: Linear Regression with Gradient Descent

In [ ]:
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

# make_regression creates a random linear regression problem
X, y = make_regression(n_samples=200, n_features=3, noise=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

X_train.shape, X_test.shape

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
for i in range(3):
    ax[i].scatter(X_train[:, i], y_train, c='b', s=10, label="train")
    ax[i].scatter(X_test[:, i], y_test, c='g', s=10, label="test")
    ax[i].set_xlabel(f"feature {i}")
ax[0].set_ylabel("y")
ax[0].legend()
fig.tight_layout()
plt.show()

scikit-learn gives us NumPy arrays of 64-bit floats. PyTorch models normally work in 32-bit floats (`float32`), so we convert now.

In [ ]:
X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
y_test_t  = torch.tensor(y_test,  dtype=torch.float32)

X_train_t.shape, y_train_t.shape

### What steps do we need?

* Using the current parameters, calculate $\hat{y} = Xw + b$
* Calculate the loss, the mean squared error: $J = \frac{1}{n}\sum_i (\hat{y}_i - y_i)^2$
* Update the parameters using gradient descent: $w \leftarrow w - \alpha\frac{\partial J}{\partial w}$, and the same for $b$

Let's initialize the parameters. We have 3 features, so 3 weights and 1 bias. `requires_grad=True` asks PyTorch to compute gradients for them.

In [ ]:
n_features = X_train_t.shape[1]

w = torch.zeros(n_features, requires_grad=True)
b = torch.zeros(1, requires_grad=True)

w, b

In [ ]:
X_train_t.shape, w.shape, (X_train_t @ w).shape

In [ ]:
def forward(X, w, b):
    return X @ w + b

def mse_loss(y_pred, y_true):
    return ((y_pred - y_true) ** 2).mean()

### Checking autograd against calculus

For linear regression we can work out the gradient by hand:

$$\frac{\partial J}{\partial w} = \frac{2}{n} X^\top (\hat{y} - y), \qquad \frac{\partial J}{\partial b} = \frac{2}{n} \sum_i (\hat{y}_i - y_i)$$

Let's confirm that autograd gives the same answer.

In [ ]:
loss = mse_loss(forward(X_train_t, w, b), y_train_t)
loss.backward()

with torch.no_grad():
    n = len(y_train_t)
    residual = forward(X_train_t, w, b) - y_train_t
    grad_w_by_hand = 2 / n * X_train_t.T @ residual
    grad_b_by_hand = 2 / n * residual.sum()

print("autograd  dJ/dw:", w.grad)
print("by hand   dJ/dw:", grad_w_by_hand)
print("autograd  dJ/db:", b.grad.item())
print("by hand   dJ/db:", grad_b_by_hand.item())
print("match:", torch.allclose(w.grad, grad_w_by_hand, rtol=1e-4))

w.grad.zero_(); b.grad.zero_();   # reset before training

Autograd matches the calculus. From here on we let autograd do it, which pays off once models get too complicated to differentiate by hand.

### The training loop

In [ ]:
learning_rate = 0.05
n_epochs = 100
loss_history = []

for epoch in range(n_epochs):
    y_pred = forward(X_train_t, w, b)       # forward pass
    loss = mse_loss(y_pred, y_train_t)      # loss
    loss.backward()                         # backward pass

    with torch.no_grad():                   # update
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad
    w.grad.zero_()
    b.grad.zero_()

    loss_history.append(loss.item())
    if epoch % 10 == 0:
        print(f"epoch {epoch:3d}   MSE = {loss.item():10.2f}")

In [ ]:
plt.plot(loss_history)
plt.xlabel("epoch")
plt.ylabel("training MSE")
plt.yscale("log")
plt.title("Gradient descent: linear regression")
plt.show()

In [ ]:
w, b

In [ ]:
print("learned w:", w.detach().numpy())
print("learned b:", b.item())

### Evaluating the model

We don't need gradients when making predictions, so we wrap them in `torch.no_grad()`.

Note: scikit-learn's metrics take the arguments in the order `(y_true, y_pred)`. For R², the order matters.

In [ ]:
with torch.no_grad():
    train_preds = forward(X_train_t, w, b).numpy()
    test_preds  = forward(X_test_t,  w, b).numpy()

print(f"Train MSE : {mean_squared_error(y_train, train_preds):.2f}")
print(f"Test  MSE : {mean_squared_error(y_test,  test_preds):.2f}")
print(f"Train R^2 : {r2_score(y_train, train_preds):.3f}")
print(f"Test  R^2 : {r2_score(y_test,  test_preds):.3f}")

### Comparison with scikit-learn

scikit-learn solves linear regression exactly (no iterations). Our gradient descent should land on essentially the same coefficients.

In [ ]:
from sklearn.linear_model import LinearRegression

sk_model = LinearRegression()
sk_model.fit(X_train, y_train)

print("sklearn coef:", sk_model.coef_, "  intercept:", sk_model.intercept_)
print("ours    coef:", w.detach().numpy(), "  intercept:", b.item())

print(f"\nsklearn Test R^2 : {r2_score(y_test, sk_model.predict(X_test)):.3f}")

> **Try it:** rerun the training cells with `learning_rate = 0.5`, then `1.1`. What happens to the loss? A learning rate that's too large makes gradient descent overshoot the minimum and diverge.

# Part 2: Logistic Regression with Gradient Descent, and with Stochastic Gradient Descent

### What steps do we need *for logistic regression*?

* Using the current parameters, calculate a probability: $\hat{y} = \sigma(Xw + b) = \dfrac{1}{1 + e^{-(Xw+b)}}$
* Calculate the loss, the **binary cross-entropy** (also called log loss):
$$J = \frac{1}{n}\sum_i \Big[-y_i \log(\hat{y}_i) - (1 - y_i)\log(1 - \hat{y}_i)\Big]$$
* Update the parameters using gradient descent, exactly as before

Only the forward pass and the loss change. The training loop is the same.

In [ ]:
from sklearn.datasets import load_breast_cancer

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=0.8, stratify=y, random_state=123)

# Standardize using statistics from the TRAINING set only
mean = X_train.mean(axis=0)
std = X_train.std(axis=0)
X_train = (X_train - mean) / std
X_test = (X_test - mean) / std

X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
y_test_t  = torch.tensor(y_test,  dtype=torch.float32)

X_train_t.shape, y_train_t.shape

In [ ]:
def sigmoid(z):
    return 1 / (1 + torch.exp(-z))

def forward_logistic(X, w, b):
    return sigmoid(X @ w + b)

def bce_loss(y_pred, y_true, eps=1e-7):
    # clamp keeps us away from log(0) if a prediction hits exactly 0 or 1
    y_pred = y_pred.clamp(eps, 1 - eps)
    return (-y_true * torch.log(y_pred) - (1 - y_true) * torch.log(1 - y_pred)).mean()

In [ ]:
z = torch.linspace(-8, 8, 200)
plt.plot(z, sigmoid(z))
plt.axhline(0.5, color="gray", ls=":")
plt.title(r"Sigmoid: $\sigma(z) = 1 / (1 + e^{-z})$")
plt.xlabel("z")
plt.show()

## Batch gradient descent

First the same approach as Part 1: every update uses **all** training examples. We'll track both the training and test loss each epoch.

In [ ]:
n_features = X_train_t.shape[1]
w = torch.zeros(n_features, requires_grad=True)
b = torch.zeros(1, requires_grad=True)

learning_rate = 0.1
n_epochs = 100

for epoch in range(n_epochs):
    y_pred = forward_logistic(X_train_t, w, b)
    loss = bce_loss(y_pred, y_train_t)
    loss.backward()
    with torch.no_grad():
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad
    w.grad.zero_()
    b.grad.zero_()

    if epoch % 10 == 0:
        with torch.no_grad():
            test_loss = bce_loss(forward_logistic(X_test_t, w, b), y_test_t)
        print(f"epoch {epoch:3d}   train loss = {loss.item():.4f}   test loss = {test_loss.item():.4f}")

## Stochastic and mini-batch gradient descent

Batch gradient descent looks at the whole dataset before taking a single step. With millions of examples, that's slow. Instead we can estimate the gradient from a small random subset:

* **Stochastic gradient descent (SGD)**: update after **every single example**. Each step is cheap but noisy.
* **Mini-batch gradient descent**: update after a small batch (say 32 examples). This is the standard in deep learning: much cheaper than full batch, much less noisy than one example, and efficient on GPUs.

One **epoch** is still one full pass through the training data. With batch size $B$ and $n$ examples, an epoch contains $\lceil n/B \rceil$ updates instead of 1.

The key ingredient is shuffling the data at the start of every epoch, so each batch is a random sample. `torch.randperm(n)` gives a random ordering of the indices `0, ..., n-1`.

In [ ]:
def minibatches(X, y, batch_size, shuffle=True):
    # Yield (X_batch, y_batch) pairs covering the whole dataset once
    n = len(X)
    idx = torch.randperm(n) if shuffle else torch.arange(n)
    for start in range(0, n, batch_size):
        batch_idx = idx[start:start + batch_size]
        yield X[batch_idx], y[batch_idx]

# Quick look: batch shapes for a batch size of 128
for Xb, yb in minibatches(X_train_t, y_train_t, batch_size=128):
    print(Xb.shape, yb.shape)

Now a training function that takes `batch_size` as a parameter. Setting `batch_size` to the full dataset size gives batch gradient descent; `batch_size=1` gives SGD.

In [ ]:
def train_logistic(batch_size, learning_rate, n_epochs, seed=0):
    torch.manual_seed(seed)
    w = torch.zeros(X_train_t.shape[1], requires_grad=True)
    b = torch.zeros(1, requires_grad=True)
    history = []
    n_updates = 0

    for epoch in range(n_epochs):
        for Xb, yb in minibatches(X_train_t, y_train_t, batch_size):
            loss = bce_loss(forward_logistic(Xb, w, b), yb)
            loss.backward()
            with torch.no_grad():
                w -= learning_rate * w.grad
                b -= learning_rate * b.grad
            w.grad.zero_()
            b.grad.zero_()
            n_updates += 1

        # loss on the full training set at the end of each epoch
        with torch.no_grad():
            history.append(bce_loss(forward_logistic(X_train_t, w, b), y_train_t).item())

    return w, b, history, n_updates

In [ ]:
n_train = len(X_train_t)
runs = {
    "Batch GD (B = all)": dict(batch_size=n_train, learning_rate=0.1),
    "Mini-batch (B = 32)": dict(batch_size=32, learning_rate=0.1),
    "SGD (B = 1)": dict(batch_size=1, learning_rate=0.01),
}

results = {}
for name, cfg in runs.items():
    w_r, b_r, hist, n_updates = train_logistic(n_epochs=30, **cfg)
    results[name] = (w_r, b_r)
    plt.plot(range(1, 31), hist, marker=".", label=f"{name}: {n_updates} updates")

plt.xlabel("epoch")
plt.ylabel("training loss (full training set)")
plt.yscale("log")
plt.legend()
plt.title("Same number of epochs, very different numbers of updates")
plt.show()

After the same number of passes through the data, mini-batch and stochastic gradient descent are far ahead, because they took hundreds or thousands of steps where batch gradient descent took 30. SGD needed a smaller learning rate, since a single example gives a noisy estimate of the gradient.

### Making predictions

The model outputs a probability. To get a class label we threshold at 0.5.

In [ ]:
from sklearn.metrics import accuracy_score

w, b = results["Mini-batch (B = 32)"]

with torch.no_grad():
    train_preds = (forward_logistic(X_train_t, w, b) > 0.5).int().numpy()
    test_preds  = (forward_logistic(X_test_t,  w, b) > 0.5).int().numpy()

print("predicted:", test_preds[:10])
print("actual:   ", y_test[:10])
print(f"\nTrain Accuracy : {accuracy_score(y_train, train_preds):.3f}")
print(f"Test  Accuracy : {accuracy_score(y_test,  test_preds):.3f}")

In [ ]:
from sklearn.linear_model import LogisticRegression

sk_model = LogisticRegression().fit(X_train, y_train)
print(f"sklearn Train Accuracy : {accuracy_score(y_train, sk_model.predict(X_train)):.3f}")
print(f"sklearn Test  Accuracy : {accuracy_score(y_test,  sk_model.predict(X_test)):.3f}")

# Part 3: Neural Network for Regression

A linear model can only draw straight lines (or flat planes). A neural network stacks several linear layers with a **nonlinear activation function** between them, which lets it fit curved relationships.

For a network with two hidden layers:

$$h_1 = \text{ReLU}(X W_1^\top + b_1), \quad h_2 = \text{ReLU}(h_1 W_2^\top + b_2), \quad \hat{y} = h_2 W_3^\top + b_3$$

Everything else (MSE loss, gradient descent) is unchanged from Part 1.

## The data: California housing

Each row is a district in California (1990 census). The 8 features include median income, house age, and location; the target is the median house value in units of $100,000.

In [ ]:
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing()
print(housing.feature_names)

X_train, X_test, y_train, y_test = train_test_split(
    housing.data, housing.target, test_size=0.2, random_state=42)

mean = X_train.mean(axis=0)
std = X_train.std(axis=0)
X_train = (X_train - mean) / std
X_test = (X_test - mean) / std

X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
y_test_t  = torch.tensor(y_test,  dtype=torch.float32)

n_samples, n_features = X_train_t.shape
n_samples, n_features

First a baseline: how well does a plain linear model do?

In [ ]:
lin = LinearRegression()
lin.fit(X_train, y_train)

print("Linear regression:")
print(f"Test MSE: {mean_squared_error(y_test, lin.predict(X_test)):.3f}")
print(f"Test R^2: {r2_score(y_test, lin.predict(X_test)):.3f}")

## Initializing the weights

We store the network as a list of `[W, b]` pairs, one per layer. Following PyTorch's convention, a layer with `n_in` inputs and `n_out` outputs has `W` of shape `(n_out, n_in)` and `b` of shape `(n_out,)`.

How we choose the initial values matters:

* Starting all weights at zero (as we did for linear regression) doesn't work: every hidden unit would compute the same thing and receive the same update, forever.
* Weights that are too large make the activations and gradients blow up as they pass through layers; too small, and they shrink toward zero.

A good default for ReLU networks is to draw weights from a normal distribution with standard deviation $\sqrt{2 / n_{in}}$, and set biases to zero (*He (Kaiming) initialization*). This keeps the size of the signal roughly constant from layer to layer.

We use a `torch.Generator` with a fixed seed so the initialization is reproducible, and each layer gets different random numbers.

In [ ]:
def init_params(layer_sizes, n_inputs, seed=42):
    gen = torch.Generator().manual_seed(seed)
    params = []
    n_in = n_inputs
    for n_out in layer_sizes:
        W = torch.randn(n_out, n_in, generator=gen) * (2 / n_in) ** 0.5
        b = torch.zeros(n_out)
        W.requires_grad_(True)
        b.requires_grad_(True)
        params.append([W, b])
        n_in = n_out
    return params

params = init_params([64, 64, 1], n_features)
for W, b in params:
    print("W:", tuple(W.shape), "  b:", tuple(b.shape))

print("total parameters:", sum(W.numel() + b.numel() for W, b in params))

## The activation function: ReLU

$\text{ReLU}(z) = \max(0, z)$. It passes positive values through unchanged and sets negative values to zero.

In [ ]:
def relu(z):
    return torch.clamp(z, min=0)

relu(torch.tensor([[-1., 0., 1., -2., 4.],
                   [ 1., 2., -4., -5., 7.]]))

## Forward pass and loss

A layer computes `x @ W.T + b` and applies an activation. Hidden layers use ReLU. The output layer has no activation, since a regression output can be any real number.

In [ ]:
def linear_layer(x, W, b, activation=None):
    out = x @ W.T + b
    return activation(out) if activation is not None else out

def forward_nn(params, X, output_activation=None):
    h = X
    for W, b in params[:-1]:
        h = linear_layer(h, W, b, relu)          # hidden layers
    W, b = params[-1]
    out = linear_layer(h, W, b, output_activation)  # output layer
    return out.squeeze(-1)                       # (n, 1) -> (n,)

preds = forward_nn(params, X_train_t)
preds.shape, mse_loss(preds, y_train_t)

## Training with full-batch gradient descent

The update loops over all `[W, b]` pairs. This is the same three-line update as before, just repeated for every layer.

In [ ]:
def sgd_update(params, learning_rate):
    with torch.no_grad():
        for W, b in params:
            W -= learning_rate * W.grad
            b -= learning_rate * b.grad
            W.grad.zero_()
            b.grad.zero_()

def train_full_batch(params, X, y, learning_rate, n_epochs, print_every=100):
    history = []
    for epoch in range(n_epochs):
        loss = mse_loss(forward_nn(params, X), y)
        loss.backward()
        sgd_update(params, learning_rate)
        history.append(loss.item())
        if epoch % print_every == 0:
            print(f"epoch {epoch:4d}   MSE = {loss.item():.4f}")
    return history

In [ ]:
params = init_params([64, 64, 1], n_features)
history_full = train_full_batch(params, X_train_t, y_train_t,
                                learning_rate=0.1, n_epochs=1000)

In [ ]:
def report_regression(params, label=""):
    with torch.no_grad():
        train_preds = forward_nn(params, X_train_t).numpy()
        test_preds  = forward_nn(params, X_test_t).numpy()
    print(f"{label}Train MSE: {mean_squared_error(y_train, train_preds):.3f}   "
          f"Test MSE: {mean_squared_error(y_test, test_preds):.3f}   "
          f"Test R^2: {r2_score(y_test, test_preds):.3f}")
    return test_preds

test_preds = report_regression(params, "Neural net (full batch)   ")

## Training with mini-batches

Now combine the neural network with the mini-batch idea from Part 2. With ~16,500 training rows and batches of 32, each epoch makes over 500 updates, so far fewer epochs are needed.

In [ ]:
def train_minibatch(params, X, y, learning_rate, n_epochs, batch_size=32,
                    loss_fn=mse_loss, output_activation=None):
    history = []
    for epoch in range(n_epochs):
        batch_losses = []
        for Xb, yb in minibatches(X, y, batch_size):
            loss = loss_fn(forward_nn(params, Xb, output_activation), yb)
            loss.backward()
            sgd_update(params, learning_rate)
            batch_losses.append(loss.item())
        history.append(np.mean(batch_losses))
        print(f"epoch {epoch:3d}   mean batch loss = {history[-1]:.4f}")
    return history

In [ ]:
torch.manual_seed(0)
params = init_params([64, 64, 1], n_features)
history_mb = train_minibatch(params, X_train_t, y_train_t,
                             learning_rate=0.005, n_epochs=20)

> **Why a smaller learning rate than for full-batch training?** A handful of districts have extreme feature values: after standardizing, the largest "average occupancy" is more than 100 standard deviations above the mean. When one of those lands in a batch of 32, it produces a huge gradient. In full-batch training it gets averaged with 16,000 ordinary rows, but in a small batch it can throw the weights so far that the loss becomes `nan`. **Try it:** rerun the cell above with `learning_rate=0.01`.

In [ ]:
test_preds = report_regression(params, "Neural net (mini-batch)   ")

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].plot(np.arange(1, len(history_full) + 1), history_full, label="full batch (1 update / epoch)")
ax[0].plot(np.arange(1, len(history_mb) + 1), history_mb, label="mini-batch (~516 updates / epoch)")
ax[0].set_xscale("log"); ax[0].set_xlabel("epoch"); ax[0].set_ylabel("training MSE")
ax[0].set_ylim(0, 1.5); ax[0].legend()

ax[1].scatter(y_test, test_preds, s=3, alpha=0.3)
ax[1].plot([0, 5], [0, 5], "r--")
ax[1].set_xlabel("actual value ($100k)"); ax[1].set_ylabel("predicted value ($100k)")
ax[1].set_title("Test set predictions")
fig.tight_layout()
plt.show()

The neural network clearly beats the linear model, because it can capture nonlinear effects (for example, how location interacts with income). Notice also the horizontal band at 5 in the actual values: the dataset caps house values at $500,000.

## Optional: backpropagation by hand

What is `loss.backward()` actually doing for a neural network? It applies the chain rule, layer by layer, from the loss back to the inputs. That's **backpropagation**. For a network with one hidden layer,

$$Z_1 = X W_1^\top + b_1, \quad A_1 = \text{ReLU}(Z_1), \quad \hat{y} = A_1 W_2^\top + b_2, \quad J = \tfrac{1}{n}\textstyle\sum (\hat{y} - y)^2,$$

the gradients flow backwards like this:

In [ ]:
torch.manual_seed(0)
Xs, ys = X_train_t[:50], y_train_t[:50]     # small batch
small = init_params([16, 1], n_features)
(W1, b1), (W2, b2) = small

# Forward pass, keeping the intermediate values
Z1 = Xs @ W1.T + b1
A1 = relu(Z1)
y_hat = (A1 @ W2.T + b2).squeeze(-1)
loss = mse_loss(y_hat, ys)

# Autograd
loss.backward()

# By hand, using the chain rule from the output backwards
with torch.no_grad():
    n = len(ys)
    d_yhat = 2 * (y_hat - ys) / n            # dJ/dy_hat        (n,)
    dW2 = d_yhat[None, :] @ A1               # dJ/dW2           (1, 16)
    db2 = d_yhat.sum(dim=0, keepdim=True)    # dJ/db2           (1,)
    dA1 = d_yhat[:, None] @ W2               # dJ/dA1           (n, 16)
    dZ1 = dA1 * (Z1 > 0)                     # ReLU passes gradient only where Z1 > 0
    dW1 = dZ1.T @ Xs                         # dJ/dW1           (16, 8)
    db1 = dZ1.sum(dim=0)                     # dJ/db1           (16,)

for name, by_hand, auto in [("W1", dW1, W1.grad), ("b1", db1, b1.grad),
                            ("W2", dW2, W2.grad), ("b2", db2, b2.grad)]:
    print(f"{name}: match = {torch.allclose(by_hand, auto, atol=1e-6)}")

Same answers. Doing this by hand for every new architecture would be tedious and error-prone, which is exactly why autograd is so valuable.

# Part 4: Neural Network for Classification

Going from regression to binary classification takes three changes, the same three we made going from linear to logistic regression:

| | Regression | Binary classification |
|---|---|---|
| Output activation | none (identity) | sigmoid, giving a probability |
| Loss | mean squared error | binary cross-entropy |
| Metric | MSE, R² | accuracy |

The hidden layers, initialization, and training loop are all unchanged. Back to the breast cancer data:

In [ ]:
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=0.8, stratify=y, random_state=123)

mean = X_train.mean(axis=0)
std = X_train.std(axis=0)
X_train = (X_train - mean) / std
X_test = (X_test - mean) / std

X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
y_test_t  = torch.tensor(y_test,  dtype=torch.float32)

n_samples, n_features = X_train_t.shape
n_samples, n_features

In [ ]:
torch.manual_seed(0)
params = init_params([16, 8, 1], n_features)

history_clf = train_minibatch(params, X_train_t, y_train_t,
                              learning_rate=0.05, n_epochs=20, batch_size=16,
                              loss_fn=bce_loss, output_activation=sigmoid)

In [ ]:
with torch.no_grad():
    train_proba = forward_nn(params, X_train_t, sigmoid)
    test_proba  = forward_nn(params, X_test_t,  sigmoid)

train_preds = (train_proba > 0.5).int().numpy()
test_preds  = (test_proba  > 0.5).int().numpy()

print(f"Train loss : {bce_loss(train_proba, y_train_t):.4f}   Test loss : {bce_loss(test_proba, y_test_t):.4f}")
print(f"Train Accuracy : {accuracy_score(y_train, train_preds):.3f}")
print(f"Test  Accuracy : {accuracy_score(y_test,  test_preds):.3f}")

On this small, nearly linearly separable dataset, the neural net performs about the same as logistic regression. Neural networks earn their keep on problems with more complex structure, like the housing data above, and images in notebook 3.

> **A note on numerical stability:** computing `sigmoid` and then `log` separately can lose precision when the sigmoid output is extremely close to 0 or 1 (that's why we needed `clamp`). In practice, the model outputs the raw score (the "logit", before the sigmoid), and a combined function computes the loss directly from it in a numerically stable way. In PyTorch that's `nn.BCEWithLogitsLoss`, which we'll use in the next notebook.

# Summary: what we did by hand, and what PyTorch will do for us

| By hand in this notebook | PyTorch tool (next notebook) |
|---|---|
| list of `[W, b]` tensors with `requires_grad=True` | `nn.Linear`, `nn.Sequential`, `nn.Module` |
| He initialization written out | built into the layers (and `nn.init`) |
| `relu`, `sigmoid` functions | `nn.ReLU`, `nn.Sigmoid`, `torch.nn.functional` |
| `mse_loss`, `bce_loss` | `nn.MSELoss`, `nn.BCEWithLogitsLoss`, `nn.CrossEntropyLoss` |
| `sgd_update` loop (subtract gradient, zero gradient) | `torch.optim.SGD` / `Adam`: `optimizer.step()`, `optimizer.zero_grad()` |
| `minibatches` generator with `randperm` | `TensorDataset` + `DataLoader(shuffle=True)` |

The overall recipe stays the same: **forward → loss → backward → update**, repeated over mini-batches and epochs.